In [ ]:
import uxarray as ux
import xarray as xr
import numpy as np
import geocat.datafiles as gdf
import geocat.comp as gc


In [3]:
data_path = 'netcdf_files/e2p3b09.F2000climo.ne30pg3.ctl002.cam.h0.0005-01.zonal_mpsi_subset.nc'
grid_path = 'netcdf_files/ne30pg3_scrip_170604.nc'
uxds = ux.open_dataset(gdf.get(grid_path), gdf.get(data_path)).drop('lat') #dropping lat to avoid using it instead of the UXarray grid

uxds

/var/folders/dd/_xm_pbpd3flgbvbnt7qhd70snnbpj_/T/ipykernel_69743/1643203269.py:3: DeprecationWarning: dropping variables using `drop` is deprecated; use drop_vars.
  uxds = ux.open_dataset(gdf.get(grid_path), gdf.get(data_path)).drop('lat') #dropping lat to avoid using it instead of the UXarray grid


<xarray.UxDataset> Size: 6MB
Dimensions:  (time: 1, n_face: 48600, lev: 32, ilev: 33)
Coordinates:
  * lev      (lev) float64 256B 3.643 7.595 14.36 24.61 ... 957.5 976.3 992.6
  * ilev     (ilev) float64 264B 2.255 5.032 10.16 18.56 ... 967.5 985.1 1e+03
  * time     (time) object 8B 0005-02-01 00:00:00
Dimensions without coordinates: n_face
Data variables:
    PS       (time, n_face) float32 194kB ...
    V        (time, lev, n_face) float32 6MB ...
    hyam     (lev) float64 256B ...
    hybm     (lev) float64 256B ...
    hyai     (ilev) float64 264B ...
    hybi     (ilev) float64 264B ...

In [4]:
downsamplePS = uxds['PS'].remap.nearest_neighbor(
    destination_grid=ux.open_grid('grid.nc'), remap_to="face centers"
)
downsampleV = uxds['V'].remap.nearest_neighbor(
    destination_grid=ux.open_grid('grid.nc'), remap_to="face centers"
)

In [5]:
ds_combined = xr.Dataset({
    "V":  downsampleV,
    "PS": downsamplePS
})

In [7]:
ds_combined['hyai'] = uxds['hyai'].isel(ilev=slice(0, 10))
ds_combined['hybi'] = uxds['hybi'].isel(ilev=slice(0, 10))
ds_combined = ds_combined.isel(ilev=slice(0, 10))
ds_combined

<xarray.Dataset> Size: 1kB
Dimensions:  (lev: 32, time: 1, n_face: 4, ilev: 10)
Coordinates:
  * lev      (lev) float64 256B 3.643 7.595 14.36 24.61 ... 957.5 976.3 992.6
  * time     (time) object 8B 0005-02-01 00:00:00
  * ilev     (ilev) float64 80B 2.255 5.032 10.16 18.56 ... 56.24 66.8 80.7
Dimensions without coordinates: n_face
Data variables:
    V        (time, lev, n_face) float32 512B 2.111 2.241 2.181 ... 3.164 3.376
    PS       (time, n_face) float32 16B 1.011e+05 1.011e+05 1.011e+05 1.011e+05
    hyai     (ilev) float64 80B ...
    hybi     (ilev) float64 80B ...

In [8]:
ds_combined = ds_combined.isel(lev=slice(0, 10))
ds_combined['hyam'] = uxds['hyam'].isel(lev=slice(0, 10))
ds_combined['hybm'] = uxds['hybm'].isel(lev=slice(0, 10))
ds_combined['V'] = ds_combined['V'].isel(lev=slice(0, 10))
ds_combined

<xarray.Dataset> Size: 664B
Dimensions:  (lev: 10, time: 1, n_face: 4, ilev: 10)
Coordinates:
  * lev      (lev) float64 80B 3.643 7.595 14.36 24.61 ... 61.52 73.75 87.82
  * time     (time) object 8B 0005-02-01 00:00:00
  * ilev     (ilev) float64 80B 2.255 5.032 10.16 18.56 ... 56.24 66.8 80.7
Dimensions without coordinates: n_face
Data variables:
    V        (time, lev, n_face) float32 160B 2.111 2.241 2.181 ... 1.881 1.488
    PS       (time, n_face) float32 16B 1.011e+05 1.011e+05 1.011e+05 1.011e+05
    hyai     (ilev) float64 80B ...
    hybi     (ilev) float64 80B ...
    hyam     (lev) float64 80B ...
    hybm     (lev) float64 80B ...

In [ ]:
ds_combined.to_netcdf('zonal_mpsi_hybrid.nc')

In [10]:
da_ipress = gc.interpolation.interp_hybrid_to_pressure(
                ds_combined.V, ds_combined.PS, ds_combined.hyam, ds_combined.hybm, lev_dim='lev'
            )
da_ipress

<xarray.DataArray 'V' (time: 1, plev: 21, n_face: 4)> Size: 336B
dask.array<_vertical_remap, shape=(1, 21, 4), dtype=float32, chunksize=(1, 21, 4), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) object 8B 0005-02-01 00:00:00
  * plev     (plev) float32 84B 1e+05 9.25e+04 8.5e+04 ... 300.0 200.0 100.0
Dimensions without coordinates: n_face

In [ ]:
da_ipress['PS'] = ds_combined['PS'] #adds as a coordinate in DArray instead of variable in DSet, not sure if this will matter but curious to find out
da_ipress

<xarray.DataArray 'V' (time: 1, plev: 21, n_face: 4)> Size: 336B
dask.array<_vertical_remap, shape=(1, 21, 4), dtype=float32, chunksize=(1, 21, 4), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) object 8B 0005-02-01 00:00:00
  * plev     (plev) float32 84B 1e+05 9.25e+04 8.5e+04 ... 300.0 200.0 100.0
    PS       (time, n_face) float32 16B 1.011e+05 1.011e+05 1.011e+05 1.011e+05
Dimensions without coordinates: n_face

In [ ]:
da_ipress.to_netcdf('zonal_mpsi_plev.nc')